In [15]:
# Importações básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Bibliotecas especializadas
import missingno as msno
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
import category_encoders as ce

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


In [22]:
from pathlib import Path 

path = Path('/home/luma/4 PERÍODO/APRENDIZAGEM DE MÁQUINA/Projeto-aprendizagem-de-maquina-/data/Airbnb Prices in European Cities/raw')
list_of_dfs = []
for file in os.listdir(path):
    if file.endswith(".csv"):
        # Cria o caminho completo para o arquivo
        file_path = path / file
        
        # Lê o CSV
        df_temp = pd.read_csv(file_path)
        
        # OPCIONAL: Adiciona uma coluna identificando a fonte do dado
        # Ex: 'amsterdam_weekdays'
        df_temp['source_file'] = file.replace('.csv', '')
        
        # Adiciona o DataFrame lido à lista
        list_of_dfs.append(df_temp)
        
        print(f"Lido: {file}. Linhas: {len(df_temp)}")

df = pd.concat(list_of_dfs, ignore_index=True)

print("\n------------------------------")
print(f"Todos os dados foram carregados no DataFrame df.")
print(f"Total de linhas carregadas: {len(df)}")
print(f"Colunas: {df.columns.tolist()}")


Lido: berlin_weekdays.csv. Linhas: 1284
Lido: london_weekends.csv. Linhas: 5379
Lido: london_weekdays.csv. Linhas: 4614
Lido: paris_weekends.csv. Linhas: 3558
Lido: budapest_weekdays.csv. Linhas: 2074
Lido: paris_weekdays.csv. Linhas: 3130
Lido: lisbon_weekdays.csv. Linhas: 2857
Lido: budapest_weekends.csv. Linhas: 1948
Lido: athens_weekdays.csv. Linhas: 2653
Lido: barcelona_weekdays.csv. Linhas: 1555
Lido: vienna_weekends.csv. Linhas: 1799
Lido: berlin_weekends.csv. Linhas: 1200
Lido: barcelona_weekends.csv. Linhas: 1278
Lido: lisbon_weekends.csv. Linhas: 2906
Lido: rome_weekends.csv. Linhas: 4535
Lido: vienna_weekdays.csv. Linhas: 1738
Lido: athens_weekends.csv. Linhas: 2627
Lido: amsterdam_weekdays.csv. Linhas: 1103
Lido: amsterdam_weekends.csv. Linhas: 977
Lido: rome_weekdays.csv. Linhas: 4492

------------------------------
Todos os dados foram carregados no DataFrame df.
Total de linhas carregadas: 51707
Colunas: ['Unnamed: 0', 'realSum', 'room_type', 'room_shared', 'room_private

In [23]:
# 1. Informações básicas do dataset
print("\n1️⃣ INFORMAÇÕES GERAIS:")
print(f"Shape: {df.shape}")
print(f"Memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


1️⃣ INFORMAÇÕES GERAIS:
Shape: (51707, 21)
Memória: 13.50 MB


In [24]:
# 2. Tipos de dados
print("\n2️⃣ TIPOS DE DADOS:")
print(df.dtypes)


2️⃣ TIPOS DE DADOS:
Unnamed: 0                      int64
realSum                       float64
room_type                      object
room_shared                      bool
room_private                     bool
person_capacity               float64
host_is_superhost                bool
multi                           int64
biz                             int64
cleanliness_rating            float64
guest_satisfaction_overall    float64
bedrooms                        int64
dist                          float64
metro_dist                    float64
attr_index                    float64
attr_index_norm               float64
rest_index                    float64
rest_index_norm               float64
lng                           float64
lat                           float64
source_file                    object
dtype: object


In [25]:
# 3. Primeiras linhas
print("\n3️⃣ PRIMEIRAS 5 LINHAS:")
display(df.head())


3️⃣ PRIMEIRAS 5 LINHAS:


,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,source_file
0,0,185.799757,Private room,False,True,2.0,True,0,0,10.0,98.0,1,3.582211,0.174706,105.063708,16.019042,148.941114,30.710638,13.42344,52.49150,berlin_weekdays
1,1,194.914462,Private room,False,True,5.0,False,0,1,9.0,86.0,1,3.525410,0.511922,75.339529,11.487002,106.442356,21.947685,13.46800,52.51900,berlin_weekdays
2,2,176.217631,Private room,False,True,2.0,False,0,0,9.0,91.0,1,3.801713,0.281397,73.669176,11.232324,105.440205,21.741048,13.47096,52.51527,berlin_weekdays
3,3,207.768533,Private room,False,True,3.0,True,0,0,10.0,97.0,1,0.982408,0.705573,133.187409,20.307057,198.233362,40.874362,13.42281,52.53139,berlin_weekdays
4,4,150.743199,Private room,False,True,2.0,False,0,0,10.0,99.0,1,8.869697,2.187188,39.860151,6.077469,50.996308,10.515090,13.52440,52.47842,berlin_weekdays


In [26]:
# 4. Informações detalhadas
print("\n4️⃣ INFO DETALHADA:")
df.info()


4️⃣ INFO DETALHADA:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  51707 non-null  int64  
 1   realSum                     51707 non-null  float64
 2   room_type                   51707 non-null  object 
 3   room_shared                 51707 non-null  bool   
 4   room_private                51707 non-null  bool   
 5   person_capacity             51707 non-null  float64
 6   host_is_superhost           51707 non-null  bool   
 7   multi                       51707 non-null  int64  
 8   biz                         51707 non-null  int64  
 9   cleanliness_rating          51707 non-null  float64
 10  guest_satisfaction_overall  51707 non-null  float64
 11  bedrooms                    51707 non-null  int64  
 12  dist                        51707 non-null  float64
 13  metro_dist

In [27]:
# 5. Estatísticas descritivas
print("\n5️⃣ ESTATÍSTICAS DESCRITIVAS:")
display(df.describe())


5️⃣ ESTATÍSTICAS DESCRITIVAS:


,Unnamed: 0,realSum,person_capacity,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat
count,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.00000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000
mean,1620.502388,279.879591,3.161661,0.291353,0.350204,9.390624,92.628232,1.15876,3.191285,0.681540,294.204105,13.423792,626.856696,22.786177,7.426068,45.671128
std,1217.380366,327.948386,1.298545,0.454390,0.477038,0.954868,8.945531,0.62741,2.393803,0.858023,224.754123,9.807985,497.920226,17.804096,9.799725,5.249263
min,0.000000,34.779339,2.000000,0.000000,0.000000,2.000000,20.000000,0.00000,0.015045,0.002301,15.152201,0.926301,19.576924,0.592757,-9.226340,37.953000
25%,646.000000,148.752174,2.000000,0.000000,0.000000,9.000000,90.000000,1.00000,1.453142,0.248480,136.797385,6.380926,250.854114,8.751480,-0.072500,41.399510
50%,1334.000000,211.343089,3.000000,0.000000,0.000000,10.000000,95.000000,1.00000,2.613538,0.413269,234.331748,11.468305,522.052783,17.542238,4.873000,47.506690
75%,2382.000000,319.694287,4.000000,1.000000,1.000000,10.000000,99.000000,1.00000,4.263077,0.737840,385.756381,17.415082,832.628988,32.964603,13.518825,51.471885
max,5378.000000,18545.450285,6.000000,1.000000,1.000000,10.000000,100.000000,10.00000,25.284557,14.273577,4513.563486,100.000000,6696.156772,100.000000,23.786020,52.641410


In [28]:
# 6. Análise de valores únicos
print("\n6️⃣ VALORES ÚNICOS POR COLUNA:")
for col in df.columns:
    unique_count = df[col].nunique()
    print(f"{col}: {unique_count} valores únicos")


6️⃣ VALORES ÚNICOS POR COLUNA:
Unnamed: 0: 5379 valores únicos
realSum: 10497 valores únicos
room_type: 3 valores únicos
room_shared: 2 valores únicos
room_private: 2 valores únicos
person_capacity: 5 valores únicos
host_is_superhost: 2 valores únicos
multi: 2 valores únicos
biz: 2 valores únicos
cleanliness_rating: 9 valores únicos
guest_satisfaction_overall: 53 valores únicos
bedrooms: 10 valores únicos
dist: 51707 valores únicos
metro_dist: 51707 valores únicos
attr_index: 51707 valores únicos
attr_index_norm: 51688 valores únicos
rest_index: 51707 valores únicos
rest_index_norm: 51688 valores únicos
lng: 23600 valores únicos
lat: 21484 valores únicos
source_file: 20 valores únicos


In [29]:
# 1. Contagem de missing values
print("\n1️⃣ MISSING VALUES POR COLUNA:")
missing_data = pd.DataFrame({
    'Coluna': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percent': (df.isnull().sum() / len(df)) * 100
})
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Percent', ascending=False)
print(missing_data)


1️⃣ MISSING VALUES POR COLUNA:
Empty DataFrame
Columns: [Coluna, Missing_Count, Missing_Percent]
Index: []


In [ ]:
# 3. Análise do mecanismo de missing values
print("\n3️⃣ ANÁLISE DO MECANISMO DE MISSING:")

In [13]:
import pandas as pd 
import os
from pathlib import Path

path = '/home/luma/4 PERÍODO/APRENDIZAGEM DE MÁQUINA/Projeto-aprendizagem-de-maquina-/data/Airbnb Prices in European Cities/raw'

for file in os.listdir(path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(path,file))
        print(f"--- {file} ---")
        print(df.columns)
        print()

--- berlin_weekdays.csv ---
Index(['Unnamed: 0', 'realSum', 'room_type', 'room_shared', 'room_private',
       'person_capacity', 'host_is_superhost', 'multi', 'biz',
       'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist',
       'metro_dist', 'attr_index', 'attr_index_norm', 'rest_index',
       'rest_index_norm', 'lng', 'lat'],
      dtype='object')

--- london_weekends.csv ---
Index(['Unnamed: 0', 'realSum', 'room_type', 'room_shared', 'room_private',
       'person_capacity', 'host_is_superhost', 'multi', 'biz',
       'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist',
       'metro_dist', 'attr_index', 'attr_index_norm', 'rest_index',
       'rest_index_norm', 'lng', 'lat'],
      dtype='object')

--- london_weekdays.csv ---
Index(['Unnamed: 0', 'realSum', 'room_type', 'room_shared', 'room_private',
       'person_capacity', 'host_is_superhost', 'multi', 'biz',
       'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist'